In [1]:
from fastai.vision.all import *
import torch.nn.utils.prune as prune
import torch.nn as nn

In [ ]:
# Import model
model_path = 'fine_tuned_plant_classifier_model_v1_base.pkl'
learn = load_learner(model_path)
# Load the DataLoaders
with open('v1_dls.pkl', 'rb') as f:
    learn.dls = pickle.load(f)
print(len(learn.dls.valid_ds))

2297


In [3]:
import torch_pruning as tp
import torch.nn as nn
import torch

In [8]:
print("Accuracy before pruning:")
learn.validate()

Accuracy before pruning:


(#2) [0.40694406628608704,0.9024814963340759]

In [ ]:
import torch
import torch.nn as nn
import torch_pruning as tp

# Eval mode
model = learn.model.eval()
# Before pruning
total_params_before = sum(p.numel() for p in model.parameters())
print(f"Total parameters before pruning: {total_params_before}")

device = next(model.parameters()).device
example_inputs = torch.randn(1, 3, 224, 224).to(device)

# Define pruning ratio and importance metric
prune_ratio = 0.4
importance = tp.importance.MagnitudeImportance()  # Correct class for L1 norm-like importance

# Collect layers to prune (conv1 of selected blocks)
target_layers = [
    #model[0][4][0].conv1,  # layer1[0]
    #model[0][4][1].conv1,  # layer1[1]
    model[0][5][0].conv1,  # layer2[0]
    model[0][6][0].conv1,  # layer3[0]
    model[0][7][0].conv1,  # layer4[0]
]

# Create list of modules to ignore for selective pruning
ignored_layers = []
for m in model.modules():
    if isinstance(m, nn.Conv2d) and m not in target_layers:
        ignored_layers.append(m)

# Build pruner
pruner = tp.pruner.MagnitudePruner(
    model=model,
    example_inputs=example_inputs,
    importance=importance,
    pruning_ratio=prune_ratio,
    iterative_steps=1,
    ignored_layers=ignored_layers
)

# Execute pruning 
pruner.step()

# Re-attach the pruned model to learner
learn.model = model

total_params_after = sum(p.numel() for p in model.parameters())
print(f"Total parameters after pruning: {total_params_after}")




Total parameters before pruning: 11721280
Total parameters after pruning: 9850608


In [9]:
print(model)

Sequential(
  (0): Sequential(
    (0): Conv2d(3, 64, kernel_size=(7, 7), stride=(2, 2), padding=(3, 3), bias=False)
    (1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (2): ReLU(inplace=True)
    (3): MaxPool2d(kernel_size=3, stride=2, padding=1, dilation=1, ceil_mode=False)
    (4): Sequential(
      (0): BasicBlock(
        (conv1): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
        (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
        (relu): ReLU(inplace=True)
        (conv2): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
        (bn2): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      )
      (1): BasicBlock(
        (conv1): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
        (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
  

In [10]:
# Fix the out_features of the final Linear layer to 34 (number of classes)
model[1][8] = nn.Linear(in_features=512, out_features=34, bias=False)

# Verify the change
print(model[1][8].out_features)


34


In [14]:
# Fine-tune after pruning
learn.model = model
learn = Learner(learn.dls, model, loss_func=CrossEntropyLossFlat(), metrics=accuracy)
learn.fine_tune(3, base_lr=1e-3)
print("Accuracy after fine-tuning:")
learn.validate()

epoch,train_loss,valid_loss,accuracy,time
0,0.839794,1.275173,0.662603,17:32


epoch,train_loss,valid_loss,accuracy,time
0,0.416559,0.779153,0.771876,21:47
1,0.271576,0.482337,0.866347,17:52
2,0.182701,0.418222,0.878537,17:55


Accuracy after fine-tuning:


(#2) [0.4182215929031372,0.8785372376441956]

In [ ]:
learn.export('v2pruned_plant_classifier.pkl')
# Save the DataLoaders separately
import pickle
with open('v2pruned_dls.pkl', 'wb') as f:
    pickle.dump(learn.dls, f)